In [ ]:
# -------------------------------------------------------
# Configuration - Local Ollama
# -------------------------------------------------------
from dotenv import load_dotenv
from openai import OpenAI
from IPython.display import Markdown, display
from ddgs import DDGS
from bs4 import BeautifulSoup
import requests

# -------------------------------------------------------
# Configuration - Local Ollama
# -------------------------------------------------------

load_dotenv()

OLLAMA_MODEL = "llama3.2"

client = OpenAI(
    base_url="http://localhost:11434/v1",
    api_key="ollama",  # Required by SDK, ignored by Ollama
    timeout=180.0
)


# -------------------------------------------------------
# Step 1: Create your system prompt
# -------------------------------------------------------

system_prompt = """
You are J.A.R.V.I.S., Tony Stark's highly intelligent
AI assistant.

You are an expert in Marvel Comics, superheroes,
villains, character histories and the MCU.

Your task is to research any Marvel character and produce
an entertaining, accurate and beautifully formatted report.

Use the following Markdown structure:

# 🦸 Character Name
Include the real name, alias and introduction.

## 📖 Origin Story
Explain the character's background and transformation.

## ⚡ Powers and Abilities
List their abilities with explanations.

## 🛡️ Weaknesses
Describe vulnerabilities and limitations.

## 🤝 Allies and Enemies
List their important allies, teams and enemies.

## 🎬 Marvel Comics vs MCU
Compare comic and movie versions where applicable.

## 🌟 Top 5 Interesting Facts
Give five surprising or entertaining facts.

## 🏆 J.A.R.V.I.S. Final Assessment
Provide an entertaining character assessment.

FORMATTING RULES:
- Use Markdown headings, tables and bullet points.
- Highlight important information in bold.
- Include suitable emojis.
- Keep paragraphs concise and entertaining.
- Prefer official Marvel sources.
- Clearly distinguish comic continuity from MCU continuity.
- Never invent unsupported facts.
- Base factual claims on the supplied research.
- Include clickable source links supporting claims.
- If information cannot be verified, explain this.
"""


# -------------------------------------------------------
# Step 2: Search the web for Marvel information
# -------------------------------------------------------

def research_character(character_name):

    print(f"🔎 Searching the web for {character_name}...")

    queries = [
        f"site:marvel.com/characters {character_name}",
        f"site:marvel.com {character_name} origin powers",
        f"{character_name} Marvel Comics MCU character biography"
    ]

    sources = []
    seen_urls = set()

    for query in queries:

        try:
            results = DDGS(timeout=15).text(
                query,
                max_results=5
            )

            for result in results:

                url = result.get("href", "")

                if (
                    not url.startswith(("https://", "http://"))
                    or url in seen_urls
                ):
                    continue

                seen_urls.add(url)

                sources.append({
                    "title": result.get("title", "Source"),
                    "url": url,
                    "description": result.get("body", "")
                })

                if len(sources) >= 6:
                    break

        except Exception as error:
            print(f"Search warning: {error}")

        if len(sources) >= 6:
            break

    print(f"📚 Found {len(sources)} research sources.")

    return sources


# -------------------------------------------------------
# Step 3: Extract website information
# -------------------------------------------------------

def scrape_webpage(url):

    try:
        response = requests.get(
            url,
            headers={
                "User-Agent": "Mozilla/5.0"
            },
            timeout=12
        )

        response.raise_for_status()

        if "text/html" not in response.headers.get(
            "Content-Type", ""
        ):
            return ""

        soup = BeautifulSoup(
            response.text,
            "html.parser"
        )

        # Remove unnecessary page elements
        for element in soup([
            "script", "style", "nav",
            "footer", "header", "noscript"
        ]):
            element.decompose()

        main_content = (
            soup.find("main")
            or soup.find("article")
            or soup.body
        )

        if main_content is None:
            return ""

        text = " ".join(main_content.stripped_strings)

        # Limit content passed into the local LLM
        return text[:2000]

    except requests.RequestException:
        return ""


# -------------------------------------------------------
# Step 4: Research and summarize any Marvel character
# -------------------------------------------------------

def summarize(character_name):

    character_name = character_name.strip()

    if not character_name:
        raise ValueError(
            "Please enter a Marvel character name."
        )

    # Research using search and web scraping
    sources = research_character(character_name)

    if not sources:
        return (
            "No web research was available. "
            "Cannot produce a verified report.",
            {}
        )

    research_text = ""

    for index, source in enumerate(sources, start=1):

        page_content = scrape_webpage(source["url"])

        research_text += f"""
SOURCE {index}
Title: {source['title']}
URL: {source['url']}
Search Summary: {source['description']}
Website Content: {page_content or 'Not available'}
-------------------------------------------
"""

    # Create research prompt
    user_prompt = f"""
Research the Marvel character: {character_name}

Use the research information provided below.

Identify the correct character and avoid confusing
different people using the same superhero identity.

Generate a detailed Marvel Character Intelligence Report
using the requested Markdown structure.

IMPORTANT:
- Use only supported information from the research.
- Do not pretend to have visited unavailable pages.
- Distinguish verified details from uncertain details.
- Include clickable source citations using supplied URLs.
- Do not invent source links or unsupported facts.

RESEARCH INFORMATION:

{research_text}
"""

    # Call local Ollama model
    print(f"🤖 Sending research to Ollama ({OLLAMA_MODEL})...")

    response = client.chat.completions.create(
        model=OLLAMA_MODEL,
        messages=[
            {
                "role": "system",
                "content": system_prompt
            },
            {
                "role": "user",
                "content": user_prompt
            }
        ],
        temperature=0.3
    )

    summary = response.choices[0].message.content or ""

    source_links = {
        source["url"]: source["title"]
        for source in sources
    }

    return summary, source_links


# -------------------------------------------------------
# Step 5: Display the character report
# -------------------------------------------------------

def display_summary(character_name):

    print(
        f"🦸 J.A.R.V.I.S. is researching "
        f"{character_name}...\n"
    )

    summary, sources = summarize(character_name)

    if not summary:
        print("No character information was found.")
        return

    # Display the report using Markdown
    display(Markdown(summary))

    # Display the research sources
    if sources:

        display(Markdown(
            "---\n### 🔗 Research Sources"
        ))

        for url, title in sources.items():
            display(Markdown(
                f"- [{title or url}]({url})"
            ))


# -------------------------------------------------------
# Step 6: Ask the user for a Marvel character
# -------------------------------------------------------

character_name = input(
    "🦸 Enter a Marvel character's name: "
).strip()

if character_name:
    display_summary(character_name)
else:
    print("Please enter a character name.")
